In [2]:
from pathlib import Path
import numpy as np, nibabel as nib, importlib.util

# Reuse the same TRAIN_MODULE, TEST_DIR you used above:
TRAIN_MODULE = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
TEST_DIR     = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")

spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MODULE)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

cfg = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                MODEL_DIR=TEST_DIR/"_tmp_models",
                                CALLBACKS_DIR=TEST_DIR/"_tmp_callbacks")
pairs, lesion_presence = seg.load_generic_dataset(cfg)

print(f"Pairs: {len(pairs)} | % non-empty masks: {lesion_presence.mean()*100:.1f}%")

# Show the first 12 pairs + mask voxel counts (raw, before crop/pad)
for (img_p, msk_p) in pairs[:12]:
    m = nib.load(str(msk_p)).get_fdata()
    print(f"IMG: {img_p.name:60s}  |  MSK: {msk_p.name:60s}  |  vox>0: {int((m>0).sum())}")


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-06 10:13:39,012 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-06 10:13:39,016 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-06 10:13:39,016 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-06 10:13:39,017 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2
2025-11-06 10:13:39,022 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-06 10:13:39,023 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=6.96GB | GPU mem tracking failed | Disk: 1244.6GB free
2025-11-06 10:13:39,047 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 276 images, 276 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_no

Strategy: MirroredStrategy


2025-11-06 10:13:57,005 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-06 10:13:57,006 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-06 10:13:57,007 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=6.90GB | GPU mem tracking failed | Disk: 1244.6GB free


Pairs: 138 | % non-empty masks: 100.0%
IMG: sub-M2029_ses-180_T1w_MNI_norm.nii.gz                         |  MSK: sub-M2029_ses-180_lesion_mask_MNI_clean.nii.gz                |  vox>0: 104897
IMG: sub-M2050_ses-208_T1w_MNI_norm.nii.gz                         |  MSK: sub-M2050_ses-208_lesion_mask_MNI_clean.nii.gz                |  vox>0: 12405
IMG: sub-M2051_ses-284_T1w_MNI_norm.nii.gz                         |  MSK: sub-M2051_ses-284_lesion_mask_MNI_clean.nii.gz                |  vox>0: 90992
IMG: sub-M2054_ses-1598_T1w_MNI_norm.nii.gz                        |  MSK: sub-M2054_ses-1598_lesion_mask_MNI_clean.nii.gz               |  vox>0: 72051
IMG: sub-M2055_ses-1469_T1w_MNI_norm.nii.gz                        |  MSK: sub-M2055_ses-1469_lesion_mask_MNI_clean.nii.gz               |  vox>0: 37145
IMG: sub-M2066_ses-325_T1w_MNI_norm.nii.gz                         |  MSK: sub-M2066_ses-325_lesion_mask_MNI_clean.nii.gz                |  vox>0: 78116
IMG: sub-M2069_ses-5818_T1w_MNI_norm.nii.g

In [4]:
import numpy as np
from pathlib import Path
import importlib.util, nibabel as nib, tensorflow as tf

# CONFIG — keep identical to your eval cell
TRAIN_MODULE = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
MODEL_PATH   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras")
TEST_DIR     = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")

# import
spec = importlib.util.spec_from_file_location("arc_seg_train", TRAIN_MODULE)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

try:
    from keras.saving import load_model
except Exception:
    from tensorflow.keras.models import load_model

m = load_model(MODEL_PATH, compile=False, custom_objects={
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
})
INPUT_SHAPE = tuple(m.input_shape[1:])

cfg = seg.DynamicTrainingConfig(DATA_DIR=TEST_DIR,
                                MODEL_DIR=TEST_DIR/"_tmp_models",
                                CALLBACKS_DIR=TEST_DIR/"_tmp_callbacks")
cfg.INPUT_SHAPE = INPUT_SHAPE
pairs, _ = seg.load_generic_dataset(cfg)

s_inter = np.float64(0.0)
s_sumy  = np.float64(0.0)
s_sump  = np.float64(0.0)

for img_p, msk_p in pairs:
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
    msk = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1])
    x = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img
    pr = m.predict(x, verbose=0)[0,...,0].astype(np.float64)  # force float64 here
    msk = msk.astype(np.float64)

    s_inter += (msk * pr).sum()
    s_sumy  += msk.sum()
    s_sump  += pr.sum()

micro_soft = (2.0 * s_inter) / (s_sumy + s_sump + 1e-12)
print("RAW sums (float64):")
print(f"  inter={s_inter:.3e}  sum_y={s_sumy:.3e}  sum_p={s_sump:.3e}")
print(f"  micro soft Dice = {micro_soft:.6f}")



Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-06 10:14:41,916 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-06 10:14:41,919 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-06 10:14:41,920 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-06 10:14:41,920 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2


Strategy: MirroredStrategy


2025-11-06 10:14:42,921 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-06 10:14:42,922 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=6.82GB | GPU mem tracking failed | Disk: 1244.6GB free
2025-11-06 10:14:42,932 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 276 images, 276 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires
2025-11-06 10:14:42,932 - SmartSOTA_Dynamic - INFO - Found 276 image files and 276 mask files
2025-11-06 10:15:00,738 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-06 10:15:00,739 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-06 10:15:00,740 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=6.84GB | GPU mem tracking failed | Disk: 1244.6GB free


RAW sums (float64):
  inter=2.386e+06  sum_y=4.650e+06  sum_p=5.185e+06
  micro soft Dice = 0.485140


In [5]:
import numpy as np

ths = np.linspace(0.1, 0.9, 17)
macro = []
# reuse already-loaded m, pairs, INPUT_SHAPE
per_case_by_thr = {float(t): [] for t in ths}
for img_p, msk_p in pairs:
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
    msk = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img
    pr = m.predict(x, verbose=0)[0,...,0].astype(np.float32)
    for t in ths:
        yhat = (pr >= t).astype(np.float32)
        inter = (msk * yhat).sum()
        denom = msk.sum() + yhat.sum() + 1e-7
        per_case_by_thr[float(t)].append((2*inter)/denom)

for t in ths:
    macro.append(np.mean(per_case_by_thr[float(t)]))
best_t = float(ths[int(np.argmax(macro))])
print("Threshold sweep (macro Dice):")
for t, mval in zip(ths, macro):
    print(f"  t={t:.2f} -> macro={mval:.4f}")
print(f"Best t={best_t:.2f} (macro={max(macro):.4f})")


Threshold sweep (macro Dice):
  t=0.10 -> macro=0.3041
  t=0.15 -> macro=0.3040
  t=0.20 -> macro=0.3040
  t=0.25 -> macro=0.3040
  t=0.30 -> macro=0.3039
  t=0.35 -> macro=0.3039
  t=0.40 -> macro=0.3039
  t=0.45 -> macro=0.3039
  t=0.50 -> macro=0.3038
  t=0.55 -> macro=0.3038
  t=0.60 -> macro=0.3038
  t=0.65 -> macro=0.3038
  t=0.70 -> macro=0.3038
  t=0.75 -> macro=0.3037
  t=0.80 -> macro=0.3037
  t=0.85 -> macro=0.3036
  t=0.90 -> macro=0.3035
Best t=0.10 (macro=0.3041)


In [6]:
import numpy as np

def dice_soft(p, y):
    p = p.astype(np.float32); y = y.astype(np.float32)
    inter = (p*y).sum(); denom = p.sum()+y.sum()+1e-7
    return (2*inter)/denom

soft_scores = []
lesion_vox  = []
for img_p, msk_p in pairs:
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
    y   = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
    x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img
    p   = m.predict(x, verbose=0)[0,...,0].astype(np.float32)
    soft_scores.append(dice_soft(p, y))
    lesion_vox.append(y.sum())

soft_scores = np.array(soft_scores); lesion_vox = np.array(lesion_vox)

print(f"Macro soft Dice: {soft_scores.mean():.4f}")
print("Percentiles (soft Dice):", np.percentile(soft_scores, [5,25,50,75,95]))
print("Lesion voxels percentiles:", np.percentile(lesion_vox, [5,25,50,75,95]))

# quick text histogram
bins = np.linspace(0,1,11)
hist, _ = np.histogram(soft_scores, bins)
for b,c in zip(zip(bins[:-1],bins[1:]), hist):
    print(f"{b[0]:.1f}-{b[1]:.1f}: {c}")


Macro soft Dice: 0.3011
Percentiles (soft Dice): [1.08672709e-04 1.50375688e-02 2.19166517e-01 5.77309191e-01
 7.61756074e-01]
Lesion voxels percentiles: [  1250.6    3640.25  23253.5   49773.25 105167.3 ]
0.0-0.1: 59
0.1-0.2: 4
0.2-0.3: 15
0.3-0.4: 4
0.4-0.5: 12
0.5-0.6: 14
0.6-0.7: 12
0.7-0.8: 16
0.8-0.9: 2
0.9-1.0: 0


In [7]:
import numpy as np

eps = 1e-7
log_size = np.log10(lesion_vox + 1)
corr = np.corrcoef(log_size, soft_scores)[0,1]
print(f"Corr(log10 lesion size, Dice) = {corr:.3f}")

# Bucket by lesion size quantiles
qs = np.quantile(lesion_vox, [0.2,0.4,0.6,0.8])
buckets = [(0, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], qs[3]), (qs[3], float('inf'))]
for lo,hi in buckets:
    m = (lesion_vox>=lo) & (lesion_vox<hi)
    print(f"vox∈[{int(lo)},{'inf' if hi==float('inf') else int(hi)}): n={m.sum():3d}, mean Dice={soft_scores[m].mean():.3f}")


Corr(log10 lesion size, Dice) = 0.618
vox∈[0,2522): n= 28, mean Dice=0.060
vox∈[2522,13074): n= 27, mean Dice=0.175
vox∈[13074,31721): n= 28, mean Dice=0.285
vox∈[31721,55269): n= 27, mean Dice=0.458
vox∈[55269,inf): n= 28, mean Dice=0.530


In [9]:
from keras.saving import load_model
import pathlib

# <<< set this to your full-model path >>>
FULL_MODEL_PATH = pathlib.Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/models/smart_sota_dynamic_20251105_165011.keras")

# Use the custom objects from your training module `seg`
custom_objects = {
    "ResidualConvBlock": seg.ResidualConvBlock,
    "VisionMambaBlock": seg.VisionMambaBlock,
    "SAM2Attention": seg.SAM2Attention,
    "CombinedLoss": seg.CombinedLoss,
    "dice_coefficient": seg.dice_coefficient,
    "dice_loss": seg.dice_loss,
    "boundary_loss": seg.boundary_loss,
}

model = load_model(FULL_MODEL_PATH, compile=False, custom_objects=custom_objects)
print("Loaded model:", FULL_MODEL_PATH.name)


Loaded model: smart_sota_dynamic_20251105_165011.keras


In [10]:
import numpy as np
from scipy.ndimage import label, generate_binary_structure

def dice_soft(p, y):
    p = p.astype(np.float32); y = y.astype(np.float32)
    inter = (p*y).sum(); denom = p.sum()+y.sum()+1e-7
    return (2*inter)/denom

def dice_hard(pbin, y):
    pbin = pbin.astype(np.float32); y = y.astype(np.float32)
    inter = (pbin*y).sum(); denom = pbin.sum()+y.sum()+1e-7
    return (2*inter)/denom

def predict_tta(model, x4d):
    preds = []
    p0 = model.predict(x4d, verbose=0)[0,...,0]
    preds.append(p0)
    for axis in [0,1,2]:
        xf = x4d.copy(); xf[0,...,0] = np.flip(x4d[0,...,0], axis=axis)
        pf = model.predict(xf, verbose=0)[0,...,0]
        preds.append(np.flip(pf, axis=axis))
    return np.mean(preds, axis=0).astype(np.float32)

def evaluate_with_tta(model, pairs, INPUT_SHAPE, th=0.5):
    softs, hards = [], []
    inter_soft = denom_soft = 0.0
    inter_hard = denom_hard = 0.0

    for img_p, msk_p in pairs:
        img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
        y   = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
        x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img

        p = predict_tta(model, x)
        pb = (p >= th).astype(np.float32)

        # macro
        softs.append(dice_soft(p, y))
        hards.append(dice_hard(pb, y))

        # micro
        inter_soft += float((p*y).sum());  denom_soft += float(p.sum()+y.sum()+1e-7)
        inter_hard += float((pb*y).sum()); denom_hard += float(pb.sum()+y.sum()+1e-7)

    macro_soft = float(np.mean(softs))
    macro_hard = float(np.mean(hards))
    micro_soft = (2*inter_soft)/denom_soft
    micro_hard = (2*inter_hard)/denom_hard
    print(f"[TTA] Macro soft={macro_soft:.4f}  Macro hard@{th}={macro_hard:.4f}")
    print(f"[TTA] Micro soft={micro_soft:.4f}  Micro hard@{th}={micro_hard:.4f}")
    return macro_soft, macro_hard, micro_soft, micro_hard

# Optional hysteresis (two-threshold) post-proc
def hysteresis_3d(p, t_low=0.20, t_high=0.50):
    strong = (p >= t_high)
    weak   = (p >= t_low)
    s = generate_binary_structure(3, 1)
    lab, n = label(weak, structure=s)
    if n == 0:
        return strong.astype(np.float32)
    keep = np.zeros_like(weak, dtype=bool)
    strong_ids = np.unique(lab[strong]); strong_ids = strong_ids[strong_ids != 0]
    for cid in strong_ids:
        keep |= (lab == cid)
    return keep.astype(np.float32)

def evaluate_hysteresis(model, pairs, INPUT_SHAPE, t_low=0.20, t_high=0.50, use_tta=True):
    softs, hards = [], []
    inter_soft = denom_soft = 0.0
    inter_hard = denom_hard = 0.0

    for img_p, msk_p in pairs:
        img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
        y   = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
        x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img

        p = predict_tta(model, x) if use_tta else model.predict(x, verbose=0)[0,...,0].astype(np.float32)
        yhat = hysteresis_3d(p, t_low=t_low, t_high=t_high)

        # macro
        inter = (yhat*y).sum(); denom = yhat.sum()+y.sum()+1e-7
        hards.append((2*inter)/denom)
        softs.append(dice_soft(p, y))

        # micro
        inter_hard += float((yhat*y).sum()); denom_hard += float(yhat.sum()+y.sum()+1e-7)
        inter_soft += float((p*y).sum());    denom_soft += float(p.sum()+y.sum()+1e-7)

    macro_hard = float(np.mean(hards))
    micro_hard = (2*inter_hard)/denom_hard
    macro_soft = float(np.mean(softs))
    micro_soft = (2*inter_soft)/denom_soft
    print(f"[HYST] t_low={t_low:.2f}, t_high={t_high:.2f}, TTA={use_tta}")
    print(f"[HYST] Macro hard={macro_hard:.4f}  Micro hard={micro_hard:.4f}")
    print(f"[HYST] Macro soft={macro_soft:.4f}  Micro soft={micro_soft:.4f}")
    return macro_hard, micro_hard, macro_soft, micro_soft

# --- run TTA baseline ---
_ = evaluate_with_tta(model, pairs, INPUT_SHAPE, th=0.5)

# --- try hysteresis on top of TTA ---
_ = evaluate_hysteresis(model, pairs, INPUT_SHAPE, t_low=0.15, t_high=0.50, use_tta=True)
_ = evaluate_hysteresis(model, pairs, INPUT_SHAPE, t_low=0.20, t_high=0.45, use_tta=True)


[TTA] Macro soft=0.2458  Macro hard@0.5=0.2474
[TTA] Micro soft=0.4056  Micro hard@0.5=0.4752
[HYST] t_low=0.15, t_high=0.50, TTA=True
[HYST] Macro hard=0.2932  Micro hard=0.4275
[HYST] Macro soft=0.2458  Micro soft=0.4056
[HYST] t_low=0.20, t_high=0.45, TTA=True
[HYST] Macro hard=0.2950  Micro hard=0.4145
[HYST] Macro soft=0.2458  Micro soft=0.4056


In [11]:
import numpy as np
from scipy.ndimage import label, generate_binary_structure, binary_closing

def dice_soft(p, y):
    p = p.astype(np.float32); y = y.astype(np.float32)
    inter = (p*y).sum(); denom = p.sum()+y.sum()+1e-7
    return (2*inter)/denom

def dice_hard(pb, y):
    pb = pb.astype(np.float32); y = y.astype(np.float32)
    inter = (pb*y).sum(); denom = pb.sum()+y.sum()+1e-7
    return (2*inter)/denom

def binarize_with_post(p, th=0.10, min_size=0, do_close=False):
    b = (p >= th)
    if do_close:
        b = binary_closing(b, structure=generate_binary_structure(3,1))
    if min_size > 0:
        s = generate_binary_structure(3,1)
        lab, n = label(b, structure=s)
        if n > 0:
            sizes = np.bincount(lab.ravel())
            keep = np.zeros_like(b, bool)
            for cid, sz in enumerate(sizes):
                if cid == 0:  # background
                    continue
                if sz >= min_size:
                    keep |= (lab == cid)
            b = keep
    return b.astype(np.float32)

def evaluate_recipe(model, pairs, INPUT_SHAPE, th, min_size, do_close):
    softs, hards = [], []
    inter_soft = denom_soft = 0.0
    inter_hard = denom_hard = 0.0
    for img_p, msk_p in pairs:
        img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1])
        y   = seg._load_and_preprocess_mask(str(msk_p), INPUT_SHAPE[:-1]).astype(np.float32)
        x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img

        p = model.predict(x, verbose=0)[0,...,0].astype(np.float32)
        pb = binarize_with_post(p, th=th, min_size=min_size, do_close=do_close)

        softs.append(dice_soft(p, y))
        hards.append(dice_hard(pb, y))
        inter_soft += float((p*y).sum());  denom_soft += float(p.sum()+y.sum()+1e-7)
        inter_hard += float((pb*y).sum()); denom_hard += float(pb.sum()+y.sum()+1e-7)

    macro_soft = float(np.mean(softs))
    macro_hard = float(np.mean(hards))
    micro_soft = (2*inter_soft)/denom_soft
    micro_hard = (2*inter_hard)/denom_hard
    return macro_soft, macro_hard, micro_soft, micro_hard

thresholds  = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
min_sizes   = [0, 500, 1500, 3000, 6000]   # voxels (~ adjust to your spacing)
closings    = [False, True]

best = None
print("Grid search (no TTA):")
for th in thresholds:
    for ms in min_sizes:
        for dc in closings:
            msd, mhd, mis, mih = evaluate_recipe(model, pairs, INPUT_SHAPE, th, ms, dc)
            print(f"th={th:0.2f} min={ms:5d} close={int(dc)} | macro_soft={msd:.4f} macro_hard={mhd:.4f} micro_hard={mih:.4f}")
            score = mhd  # optimize for macro hard; change to msd if you prefer macro soft
            if (best is None) or (score > best[0]):
                best = (score, th, ms, dc, msd, mhd, mis, mih)

print("\nBEST (by macro hard):")
score, th, ms, dc, msd, mhd, mis, mih = best
print(f"th={th:0.2f} min={ms} close={int(dc)} | macro_soft={msd:.4f} macro_hard={mhd:.4f} micro_soft={mis:.4f} micro_hard={mih:.4f}")


Grid search (no TTA):
th=0.05 min=    0 close=0 | macro_soft=0.3011 macro_hard=0.3042 micro_hard=0.4869
th=0.05 min=    0 close=1 | macro_soft=0.3011 macro_hard=0.3042 micro_hard=0.4869
th=0.05 min=  500 close=0 | macro_soft=0.3011 macro_hard=0.3037 micro_hard=0.4871
th=0.05 min=  500 close=1 | macro_soft=0.3011 macro_hard=0.3037 micro_hard=0.4871
th=0.05 min= 1500 close=0 | macro_soft=0.3011 macro_hard=0.3035 micro_hard=0.4877
th=0.05 min= 1500 close=1 | macro_soft=0.3011 macro_hard=0.3035 micro_hard=0.4877
th=0.05 min= 3000 close=0 | macro_soft=0.3011 macro_hard=0.3001 micro_hard=0.4886
th=0.05 min= 3000 close=1 | macro_soft=0.3011 macro_hard=0.3010 micro_hard=0.4886
th=0.05 min= 6000 close=0 | macro_soft=0.3011 macro_hard=0.2824 micro_hard=0.4901
th=0.05 min= 6000 close=1 | macro_soft=0.3011 macro_hard=0.2822 micro_hard=0.4900
th=0.08 min=    0 close=0 | macro_soft=0.3011 macro_hard=0.3041 micro_hard=0.4871
th=0.08 min=    0 close=1 | macro_soft=0.3011 macro_hard=0.3041 micro_hard=0

In [3]:
# === Re-test held-out set with chosen recipe (th=0.05, no morph, no size filter) ===
from pathlib import Path
import os, time, json, csv, math
import numpy as np
import tensorflow as tf

import nibabel as nib

def save_like(ref_path, array, out_path, dtype=None):
    """
    Save `array` as NIfTI using `ref_path`'s affine/header so it aligns in space.
    """
    ref = nib.load(str(ref_path))
    data = np.ascontiguousarray(array.astype(dtype if dtype is not None else np.float32))
    # keep header but avoid mutating original
    hdr = ref.header.copy()
    nii = nib.Nifti1Image(data, ref.affine, hdr)
    nib.save(nii, str(out_path))


# --- paths you can tweak ---
RUN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011")  # <- last training run
TEST_DIR  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires")
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")

# --- recipe (picked from your sweep: best macro-hard) ---
THRESH   = 0.05
MIN_SIZE = 0
DO_CLOSE = False

# --- runtime & logging dirs ---
OUT_DIR   = RUN_DIR / f"test_preds_t{THRESH:.2f}_min{MIN_SIZE}_close{int(DO_CLOSE)}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS = RUN_DIR / "test_artifacts"; ARTIFACTS.mkdir(parents=True, exist_ok=True)

# --- import your training module with custom ops ---
import importlib.util, gc
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# Ensure simple default strategy for inference
seg.strategy = tf.distribute.get_strategy()

# Locate the full Keras model from this run
keras_models = sorted((RUN_DIR / "models").glob("smart_sota_dynamic_*.keras"))
if not keras_models:
    raise FileNotFoundError("No .keras model found in RUN_DIR/models.")
MODEL_PATH = keras_models[0]
print("Loading model:", MODEL_PATH.name)

# Load model with custom objects
from keras.saving import load_model
m = load_model(
    MODEL_PATH,
    compile=False,
    custom_objects={
        "ResidualConvBlock": seg.ResidualConvBlock,
        "VisionMambaBlock": seg.VisionMambaBlock,
        "SAM2Attention": seg.SAM2Attention,
        "CombinedLoss": seg.CombinedLoss,
        "dice_coefficient": seg.dice_coefficient,
        "dice_loss": seg.dice_loss,
        "boundary_loss": seg.boundary_loss,
    },
)
INPUT_SHAPE = tuple(m.input_shape[1:])
print("Model INPUT_SHAPE:", INPUT_SHAPE)

# --- helper: simple post-processing (threshold + optional morph + min-size) ---
from scipy.ndimage import label, generate_binary_structure, binary_closing
def postproc(prob, th=THRESH, min_size=MIN_SIZE, do_close=DO_CLOSE):
    b = (prob >= th)
    if do_close:
        b = binary_closing(b, structure=generate_binary_structure(3,1))
    if min_size > 0:
        s = generate_binary_structure(3,1)
        lab, n = label(b, structure=s)
        if n > 0:
            sizes = np.bincount(lab.ravel())
            keep = np.zeros_like(b, bool)
            for cid, sz in enumerate(sizes):
                if cid and sz >= min_size:
                    keep |= (lab == cid)
            b = keep
    return b.astype(np.uint8)

# --- dataset pairing using your loader, but pointed at TEST_DIR ---
cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TEST_DIR,            # single-folder mode (images + masks together)
    IMAGES_DIR=None, MASKS_DIR=None,
    INPUT_SHAPE=INPUT_SHAPE,
    VALIDATION_SPLIT=0.10,        # unused here, but required by dataclass
    BATCH_SIZE=1,                 # inference batch size
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
)
pairs, lesion_presence = seg.load_generic_dataset(cfg)
print(f"Pairs: {len(pairs)} | % non-empty masks: {100*np.mean(lesion_presence):.1f}%")

# --- metrics helpers ---
def dice_soft(y, p, eps=1e-8):
    # y,p in [0,1]
    inter = float(np.sum(y * p, dtype=np.float64))
    denom = float(np.sum(y, dtype=np.float64) + np.sum(p, dtype=np.float64) + eps)
    return 2.0 * inter / denom, inter

def dice_hard(y, b, eps=1e-8):
    inter = float(np.sum((y > 0) & (b > 0), dtype=np.float64))
    denom = float(np.sum(y > 0, dtype=np.float64) + np.sum(b > 0, dtype=np.float64) + eps)
    return 2.0 * inter / denom, inter

# --- per-case evaluation loop ---
rows = []
sum_inter_soft = 0.0
sum_y = 0.0
sum_p = 0.0
sum_inter_hard = 0.0
sum_b = 0.0

for i, (img_p, msk_p) in enumerate(pairs, 1):
    # load & shape
    img = seg._load_and_preprocess_image(str(img_p), INPUT_SHAPE[:-1]).astype(np.float32)
    y   = seg._load_and_preprocess_mask(str(msk_p),  INPUT_SHAPE[:-1]).astype(np.float32)
    x   = np.zeros((1,*INPUT_SHAPE), np.float32); x[0,...,0] = img

    # predict
    p = m.predict(x, verbose=0)[0,...,0].astype(np.float32)
    b = postproc(p, THRESH, MIN_SIZE, DO_CLOSE)

    # metrics per case
    ds, inter_s = dice_soft(y, p)
    dh, inter_h = dice_hard(y, b)
    vy = float(np.sum(y > 0, dtype=np.float64))
    vp = float(np.sum(p >= THRESH, dtype=np.float64))  # voxels predicted positive at threshold

    # accumulate for micro
    sum_inter_soft += inter_s
    sum_y          += vy
    sum_p          += float(np.sum(p, dtype=np.float64))
    sum_inter_hard += inter_h
    sum_b          += float(np.sum(b > 0, dtype=np.float64))

    # save predictions
    base = img_p.stem.replace("_T1w", "")
    # seg._save_like(msk_p, p, OUT_DIR / f"{base}_soft.nii.gz")
    # seg._save_like(msk_p, b, OUT_DIR / f"{base}_hard.nii.gz")
    save_like(msk_p, p, OUT_DIR / f"{base}_soft.nii.gz", dtype=np.float32)
    save_like(msk_p, b, OUT_DIR / f"{base}_hard.nii.gz", dtype=np.uint8)


    rows.append({
        "case": base,
        "soft_dice": ds,
        "hard_dice": dh,
        "lesion_voxels": int(vy),
        "pred_voxels@th": int(vp),
        "inter_soft_vox": int(inter_s),
        "inter_hard_vox": int(inter_h),
    })
    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] last case soft={ds:.4f} hard@{THRESH:.2f}={dh:.4f}", flush=True)

# --- aggregate metrics ---
macro_soft = float(np.mean([r["soft_dice"] for r in rows])) if rows else 0.0
macro_hard = float(np.mean([r["hard_dice"] for r in rows])) if rows else 0.0
micro_soft = float(2.0 * sum_inter_soft / (sum_y + sum_p + 1e-8)) if (sum_y + sum_p) > 0 else 0.0
micro_hard = float(2.0 * sum_inter_hard / (sum_y + sum_b + 1e-8)) if (sum_y + sum_b) > 0 else 0.0

print("\n=== HELD-OUT RESULTS (chosen recipe) ===")
print(f"Per-case (macro) soft Dice    : {macro_soft:.4f}")
print(f"Per-case (macro) hard Dice @{THRESH:.2f}: {macro_hard:.4f}")
print(f"Global (micro) soft Dice      : {micro_soft:.4f}")
print(f"Global (micro) hard Dice @{THRESH:.2f}: {micro_hard:.4f}")
print(f"Saved predictions -> {OUT_DIR}")

# --- write artifacts ---
csv_path = ARTIFACTS / f"test_cases_t{THRESH:.2f}_min{MIN_SIZE}_close{int(DO_CLOSE)}.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

summary = {
    "threshold": THRESH,
    "min_size": MIN_SIZE,
    "closing": DO_CLOSE,
    "n_cases": len(rows),
    "macro_soft": macro_soft,
    "macro_hard": macro_hard,
    "micro_soft": micro_soft,
    "micro_hard": micro_hard,
    "run_dir": str(RUN_DIR),
    "out_dir": str(OUT_DIR),
}
json_path = ARTIFACTS / f"test_summary_t{THRESH:.2f}_min{MIN_SIZE}_close{int(DO_CLOSE)}.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote per-case CSV  ->", csv_path)
print("Wrote summary JSON  ->", json_path)

# hygiene
tf.keras.backend.clear_session(); gc.collect();


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-11-06 14:03:07,141 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-11-06 14:03:07,145 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-06 14:03:07,145 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-06 14:03:07,146 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2


Strategy: MirroredStrategy
Loading model: smart_sota_dynamic_20251105_165011.keras


2025-11-06 14:03:08,004 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-06 14:03:08,005 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=1.95GB | GPU mem tracking failed | Disk: 1244.4GB free
2025-11-06 14:03:08,015 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 276 images, 276 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_hires
2025-11-06 14:03:08,016 - SmartSOTA_Dynamic - INFO - Found 276 image files and 276 mask files


Model INPUT_SHAPE: (192, 224, 192, 1)


2025-11-06 14:03:25,647 - SmartSOTA_Dynamic - INFO - 📊 Created 138 image–mask pairs
2025-11-06 14:03:25,648 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-11-06 14:03:25,649 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.99GB | GPU mem tracking failed | Disk: 1244.4GB free


Pairs: 138 | % non-empty masks: 100.0%
[10/138] last case soft=0.7263 hard@0.05=0.7349
[20/138] last case soft=0.8046 hard@0.05=0.8015
[30/138] last case soft=0.6450 hard@0.05=0.6475
[40/138] last case soft=0.5819 hard@0.05=0.5838
[50/138] last case soft=0.5900 hard@0.05=0.5952
[60/138] last case soft=0.4064 hard@0.05=0.4233
[70/138] last case soft=0.5104 hard@0.05=0.5142
[80/138] last case soft=0.4566 hard@0.05=0.4571
[90/138] last case soft=0.0800 hard@0.05=0.0860
[100/138] last case soft=0.0001 hard@0.05=0.0000
[110/138] last case soft=0.0002 hard@0.05=0.0000
[120/138] last case soft=0.0222 hard@0.05=0.0251
[130/138] last case soft=0.0761 hard@0.05=0.0773
[138/138] last case soft=0.0228 hard@0.05=0.0226

=== HELD-OUT RESULTS (chosen recipe) ===
Per-case (macro) soft Dice    : 0.3011
Per-case (macro) hard Dice @0.05: 0.3042
Global (micro) soft Dice      : 0.4851
Global (micro) hard Dice @0.05: 0.4869
Saved predictions -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_

In [ ]:
# === Robust interactive MRI viewer (updates without %matplotlib widget) ===
# Set these:
PREDS_DIR = "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/runs/20251105_165011/test_preds_test_lores_t0.05_min0_close0"
TEST_DIR  = "/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/test_lores"

import os, re, glob, numpy as np, nibabel as nib, matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, FloatSlider, RadioButtons, Checkbox, Button, HBox, VBox, Layout, HTML, Output
from IPython.display import display, clear_output

# ---------- filename matching ----------
def _norm_key(name: str) -> str:
    n = re.sub(r"\.nii(\.gz)?$", "", name)
    n = re.sub(r"_T1w", "", n)
    n = re.sub(r"_lesion_mask(_MNI)?_clean", "", n)
    n = re.sub(r"_MNI_norm", "", n)
    n = re.sub(r"_(soft|hard)$", "", n)
    return n

def _rglob(patterns, root):
    out = []
    for p in patterns:
        out.extend(glob.glob(os.path.join(root, "**", p), recursive=True))
    return sorted(out)

def build_cases(preds_dir, test_dir):
    pred_soft = _rglob(["*_soft.nii.gz"], preds_dir)
    pred_hard = _rglob(["*_hard.nii.gz"], preds_dir)
    psoft_map = {_norm_key(os.path.basename(p)): p for p in pred_soft}
    phard_map = {_norm_key(os.path.basename(p)): p for p in pred_hard}
    gt_map  = {_norm_key(os.path.basename(p)): p for p in _rglob(["*lesion_mask*.nii.gz"], test_dir)}
    mri_map = {_norm_key(os.path.basename(p)): p for p in _rglob(["*T1w*.nii.gz"], test_dir)}
    keys = sorted(set(psoft_map) & set(phard_map) & set(gt_map) & set(mri_map))
    return [{ "key": k, "mri": mri_map[k], "gt": gt_map[k], "pred_soft": psoft_map[k], "pred_hard": phard_map[k] } for k in keys]

cases = build_cases(PREDS_DIR, TEST_DIR)
if not cases:
    raise RuntimeError("No matching cases. Check PREDS_DIR/TEST_DIR and filenames.")

# ---------- loading, caching, utils ----------
_cache = {}
def load_case(case):
    k = case["key"]
    if k not in _cache:
        mri = np.asanyarray(nib.load(case["mri"]).get_fdata())
        gt  = np.asanyarray(nib.load(case["gt"]).get_fdata())
        ps  = np.asanyarray(nib.load(case["pred_soft"]).get_fdata())
        ph  = np.asanyarray(nib.load(case["pred_hard"]).get_fdata())
        # crop to common shape
        min_shape = tuple(min(d) for d in zip(mri.shape, gt.shape, ps.shape, ph.shape))
        slc = tuple(slice(0,s) for s in min_shape)
        _cache[k] = (mri[slc], gt[slc], ps[slc], ph[slc])
    return _cache[k]

def perc_norm(x, pmin=0.5, pmax=99.5):
    a, b = np.percentile(x, [pmin, pmax])
    if b <= a: return np.zeros_like(x, np.float32)
    y = (x - a) / (b - a)
    return np.clip(y, 0, 1).astype(np.float32)

def plane(arr, axis, idx):
    return arr[:,:,idx] if axis==2 else (arr[:,idx,:] if axis==1 else arr[idx,:,:])

# ---------- widgets ----------
w_case   = Dropdown(options=[(c["key"], i) for i,c in enumerate(cases)], description="Case:", layout=Layout(width="45%"))
w_axis   = RadioButtons(options=[("axial (z)", 2), ("coronal (y)", 1), ("sagittal (x)", 0)], value=2, description="Axis:")
w_slice  = IntSlider(description="Slice:", min=0, max=10, value=0, step=1, layout=Layout(width="60%"))
w_overlay= RadioButtons(options=["None", "GT", "Pred (hard)", "Pred (soft)"], value="Pred (hard)", description="Overlay:")
w_alpha  = FloatSlider(description="Alpha:", min=0.0, max=1.0, value=0.35, step=0.05)
w_softth = FloatSlider(description="Soft Th:", min=0.0, max=1.0, value=0.05, step=0.01)
w_norm   = Checkbox(value=True, description="Normalize MRI (0.5–99.5%)")
w_redraw = Button(description="Redraw", button_style="")
w_info   = HTML(value="")
out      = Output(layout=Layout(border="1px solid #333", width="720px", height="720px"))

def refresh_slice_range(*_):
    mri, gt, ps, ph = load_case(cases[w_case.value])
    w_slice.max = mri.shape[w_axis.value]-1
    if w_slice.value > w_slice.max:
        w_slice.value = w_slice.max

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        case = cases[w_case.value]
        mri, gt, ps, ph = load_case(case)
        axc, sl = w_axis.value, w_slice.value

        bg = plane(mri, axc, sl)
        bg = perc_norm(bg) if w_norm.value else ((bg - bg.min()) / (max(1e-6, bg.max()-bg.min())))

        ov = None; title_extra = ""
        if w_overlay.value == "GT":
            ov = plane((gt>0).astype(np.float32), axc, sl); title_extra = " | GT"
        elif w_overlay.value == "Pred (hard)":
            ov = plane((ph>0).astype(np.float32), axc, sl); title_extra = " | Pred hard"
        elif w_overlay.value == "Pred (soft)":
            ov = (plane(ps, axc, sl) >= w_softth.value).astype(np.float32)
            title_extra = f" | Pred soft ≥ {w_softth.value:.02f}"

        fig, ax = plt.subplots(figsize=(6,6))
        ax.imshow(bg.T, origin="lower", cmap="gray")
        if ov is not None:
            ax.imshow(ov.T, origin="lower", alpha=w_alpha.value)
            try: ax.contour(ov.T, levels=[0.5], colors=["r"], linewidths=0.8)
            except Exception: pass
        ax.set_title(f"{case['key']} | slice {sl}{title_extra}")
        ax.axis("off")
        plt.show()

def _on_change(_):
    refresh_slice_range()
    redraw()

# wire up
for w in (w_case, w_axis):
    w.observe(_on_change, names="value")
for w in (w_slice, w_overlay, w_alpha, w_softth, w_norm):
    w.observe(redraw, names="value")
w_redraw.on_click(redraw)

# initial draw
refresh_slice_range(); redraw()

controls = VBox([
    HBox([w_case, w_axis]),
    HBox([w_slice]),
    HBox([w_overlay, w_alpha, w_softth, w_norm, w_redraw]),
    w_info
])
display(controls, out)


Output(layout=Layout(border_bottom='1px solid #333', border_left='1px solid #333', border_right='1px solid #33…